# Valve Flow Characteristics

Exploring the flow behaviour of the `Valve` block under varying pressure drop conditions.
The valve models isenthalpic flow using a standard Cv equation:

$$F = C_v \sqrt{|\Delta P|} \cdot \text{sign}(\Delta P)$$

This example is inspired by [MiniSim's ValveExample](https://github.com/Nukleon84/MiniSim/blob/master/doc/ValveExample.ipynb),
adapted to PathSim's dynamic framework.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('../pathsim_docs.mplstyle')

from pathsim import Simulation, Connection
from pathsim.blocks import Source, Scope

from pathsim_chem.process import Valve

## Pressure Drop Sweep

Hold the outlet pressure at 1 bar and ramp the inlet pressure from 1 bar to 10 bar.
Compare valves with different Cv coefficients.

In [ ]:
results = {}

for Cv in [0.5, 1.0, 2.0, 5.0]:
    valve = Valve(Cv=Cv)

    src_pin = Source(lambda t: 100000.0 + t * 10000.0)  # 1 bar -> 10 bar
    src_pout = Source(lambda t: 100000.0)                 # 1 bar fixed
    src_tin = Source(lambda t: 350.0)                     # 350 K

    sco = Scope(labels=["F", "T_out"])

    sim = Simulation(
        [src_pin, src_pout, src_tin, valve, sco],
        [
            Connection(src_pin, valve),           # P_in
            Connection(src_pout, valve[1]),        # P_out
            Connection(src_tin, valve[2]),         # T_in
            Connection(valve, sco),               # F
            Connection(valve[1], sco[1]),          # T_out
        ],
        dt=0.5,
    )

    sim.run(90)
    time, [F, T_out] = sco.read()
    dP = (100000.0 + time * 10000.0) - 100000.0  # pressure drop [Pa]
    results[f"Cv={Cv}"] = (dP / 1e5, F)  # convert dP to bar

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for label, (dP_bar, F) in results.items():
    ax.plot(dP_bar, F, label=label)

ax.set_xlabel(r"Pressure drop $\Delta P$ [bar]")
ax.set_ylabel("Flow rate $F$")
ax.set_title("Valve Characteristic Curves")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The square-root relationship between pressure drop and flow rate is clearly visible.
Doubling the Cv coefficient doubles the flow at the same pressure drop.

## Isenthalpic Expansion

The valve is an isenthalpic device — outlet temperature equals inlet temperature
regardless of pressure drop. Verify this across the full sweep.

In [ ]:
# Re-run one case and check T_out
valve = Valve(Cv=2.0)

src_pin = Source(lambda t: 100000.0 + t * 10000.0)
src_pout = Source(lambda t: 100000.0)
src_tin = Source(lambda t: 350.0)

sco = Scope(labels=["F", "T_out"])

sim = Simulation(
    [src_pin, src_pout, src_tin, valve, sco],
    [
        Connection(src_pin, valve),
        Connection(src_pout, valve[1]),
        Connection(src_tin, valve[2]),
        Connection(valve, sco),
        Connection(valve[1], sco[1]),
    ],
    dt=0.5,
)

sim.run(90)
time, [F, T_out] = sco.read()

print(f"T_out range: {T_out.min():.2f} K to {T_out.max():.2f} K")
print(f"T_in = 350.00 K (constant) -> isenthalpic confirmed")